# Motor Insurance Claims & Policy Analytics

**Purpose:** Explore the insurance portfolio, validate data quality, calculate KPIs,
study claim and premium behavior, and identify evidence-based business questions.

All findings should be interpreted as descriptive analysis, not proof of causation.

## 1. Business Understanding

The company manages customers, vehicles, policies, claims, and payments.
This analysis examines policy activity, premiums, claim volume and severity,
settlement performance, customer segments, vehicle patterns, and operational risks.

**Data model:** customers → policies; vehicles → policies; policies → claims;
customers → claims; claims → payments; policies → payments.

## 2. Import Libraries

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import load_all_data
from src import visualization as viz
from src import insights as insight_engine

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 3. Load Related Tables

In [2]:
DATA_DIR = ROOT / "data" / "cleaned"
data = load_all_data(DATA_DIR)

for table_name, df in data.items():
    print(f"{table_name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    display(df.head(3))

customers: 20,000 rows × 9 columns


,customer_id,customer_name,gender,date_of_birth,age,city,state,occupation,customer_age
0,CUST_000001,Aryan Maharaj,Male,1980-09-27,46,Bengaluru,Karnataka,Business Owner,46
1,CUST_000002,Udant Dewan,Male,1990-08-03,36,Chennai,Tamil Nadu,Sales Executive,36
2,CUST_000003,Gagan Sami,Male,1969-06-01,57,Karimnagar,Telangana,Software Engineer,57


vehicles: 25,000 rows × 9 columns


,vehicle_id,customer_id,vehicle_type,vehicle_make,vehicle_model,vehicle_year,fuel_type,vehicle_value,vehicle_age
0,VEH_000001,CUST_000216,Car,Tata,Tiago,2023,Petrol,"602,727.06",3
1,VEH_000002,CUST_011180,Car,Tata,Punch,2023,CNG,"860,471.13",3
2,VEH_000003,CUST_012391,Car,Hyundai,i20,2017,Diesel,"751,541.48",9


policies: 25,000 rows × 12 columns


,policy_id,customer_id,vehicle_id,policy_start_date,policy_end_date,policy_type,premium_amount,coverage_amount,policy_status,payment_frequency,policy_duration_days,policy_active_flag
0,POL_000001,CUST_000216,VEH_000001,2026-04-10,2027-04-09,Comprehensive,"18,799.02","572,309.02",Active,Annual,364.00,1
1,POL_000002,CUST_011180,VEH_000002,2024-09-30,2025-09-29,Comprehensive,"24,413.33","812,763.96",Renewed,Semi-Annual,364.00,0
2,POL_000003,CUST_012391,VEH_000003,2024-01-30,2025-01-29,Comprehensive,"27,496.06","738,543.68",Expired,Annual,365.00,0


claims: 10,000 rows × 20 columns


,claim_id,policy_id,customer_id,claim_date,accident_date,claim_type,accident_location,claim_amount,claim_status,approval_date,settlement_date,damage_severity,claim_processing_days,claim_settlement_days,premium_amount,coverage_amount,claim_to_premium_ratio,claim_to_coverage_ratio,total_payment_amount,paid_claim_ratio
0,CLM_000001,POL_011277,CUST_018960,2025-01-13,2025-01-07,Third Party Damage,City Road,"256,572.56",Settled,2025-01-24,2025-01-31,Low,11.00,18.00,"87,994.82","2,351,667.88",2.92,0.11,"230,757.87",0.90
1,CLM_000002,POL_020774,CUST_003532,2026-05-25,2026-05-12,Accident,Parking Area,"141,080.87",Pending,NaN,NaN,Low,NaN,NaN,"11,372.12","1,153,586.72",12.41,0.12,NaN,NaN
2,CLM_000003,POL_023289,CUST_009274,2024-04-19,2024-04-17,Third Party Damage,"Phusro, West Bengal","95,522.84",Rejected,NaN,NaN,Low,NaN,NaN,"24,636.65","726,850.71",3.88,0.13,NaN,NaN


payments: 7,000 rows × 7 columns


,payment_id,claim_id,policy_id,payment_date,payment_amount,payment_status,payment_method
0,PAY_000001,CLM_004650,POL_002355,2025-12-02,"59,275.42",Completed,Bank Transfer
1,PAY_000002,CLM_002291,POL_021490,2025-09-08,"91,119.70",Completed,UPI
2,PAY_000003,CLM_003111,POL_024892,2025-06-19,"331,851.20",Completed,Cheque


## 4. Understand Table Relationships

In [3]:
relationships = pd.DataFrame([
    ["vehicles", "customer_id", "customers", "customer_id"],
    ["policies", "customer_id", "customers", "customer_id"],
    ["policies", "vehicle_id", "vehicles", "vehicle_id"],
    ["claims", "policy_id", "policies", "policy_id"],
    ["claims", "customer_id", "customers", "customer_id"],
    ["payments", "claim_id", "claims", "claim_id"],
    ["payments", "policy_id", "policies", "policy_id"],
], columns=["Child Table", "Child Key", "Parent Table", "Parent Key"])
display(relationships)

,Child Table,Child Key,Parent Table,Parent Key
0,vehicles,customer_id,customers,customer_id
1,policies,customer_id,customers,customer_id
2,policies,vehicle_id,vehicles,vehicle_id
3,claims,policy_id,policies,policy_id
4,claims,customer_id,customers,customer_id
5,payments,claim_id,claims,claim_id
6,payments,policy_id,policies,policy_id


## 5. Basic Data Understanding

In [4]:
for name, df in data.items():
    print(f"\n--- {name.upper()} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.dtypes.rename("data_type").to_frame())
    display(df.describe(include="all").T.head(20))


--- CUSTOMERS ---
Shape: (20000, 9)
Columns: ['customer_id', 'customer_name', 'gender', 'date_of_birth', 'age', 'city', 'state', 'occupation', 'customer_age']


,data_type
customer_id,str
customer_name,str
gender,str
date_of_birth,str
age,int64
city,str
state,str
occupation,str
customer_age,int64


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,20000,20000,CUST_000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_name,20000,19276,Charvi Dara,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,20000,2,Male,12451,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_of_birth,20000,12460,1985-07-08,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,"20,000.00",NaN,NaN,NaN,43.81,15.25,18.00,31.00,44.00,57.00,70.00
city,20000,34,Kota,633,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,20000,16,Maharashtra,2368,NaN,NaN,NaN,NaN,NaN,NaN,NaN
occupation,20000,15,Software Engineer,1997,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_age,"20,000.00",NaN,NaN,NaN,43.81,15.25,18.00,31.00,44.00,57.00,70.00



--- VEHICLES ---
Shape: (25000, 9)
Columns: ['vehicle_id', 'customer_id', 'vehicle_type', 'vehicle_make', 'vehicle_model', 'vehicle_year', 'fuel_type', 'vehicle_value', 'vehicle_age']


,data_type
vehicle_id,str
customer_id,str
vehicle_type,str
vehicle_make,str
vehicle_model,str
vehicle_year,int64
fuel_type,str
vehicle_value,float64
vehicle_age,int64


...

,data_type
claim_id,str
policy_id,str
customer_id,str
claim_date,str
accident_date,str
claim_type,str
accident_location,str
claim_amount,float64
claim_status,str
approval_date,str


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
claim_id,10000,10000,CLM_000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
policy_id,10000,10000,POL_011277,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,10000,9187,CUST_009599,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
claim_date,10000,1054,2026-10-08,191,NaN,NaN,NaN,NaN,NaN,NaN,NaN
accident_date,10000,1059,2026-09-14,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
claim_type,10000,5,Accident,4838,NaN,NaN,NaN,NaN,NaN,NaN,NaN
accident_location,10000,1473,National Highway,1703,NaN,NaN,NaN,NaN,NaN,NaN,NaN
claim_amount,"10,000.00",NaN,NaN,NaN,"288,543.64","263,048.10","12,281.47","101,711.52","205,040.10","391,884.29","2,645,659.52"
claim_status,10000,4,Approved,3790,NaN,NaN,NaN,NaN,NaN,NaN,NaN
approval_date,7049,1035,2026-10-08,254,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- PAYMENTS ---
Shape: (7000, 7)
Columns: ['payment_id', 'claim_id', 'policy_id', 'payment_date', 'payment_amount', 'payment_status', 'payment_method']


,data_type
payment_id,str
claim_id,str
policy_id,str
payment_date,str
payment_amount,float64
payment_status,str
payment_method,str


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
payment_id,7000,7000,PAY_000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
claim_id,7000,7000,CLM_004650,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
policy_id,7000,7000,POL_002355,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_date,7000,1020,2026-10-08,425,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_amount,"7,000.00",NaN,NaN,NaN,"252,412.81","233,250.02","10,183.07","88,404.08","176,889.60","339,512.91","2,067,463.33"
payment_status,7000,1,Completed,7000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_method,7000,4,Bank Transfer,3098,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Data Quality Analysis

In [5]:
quality_rows = []
primary_keys = {
    "customers": "customer_id",
    "vehicles": "vehicle_id",
    "policies": "policy_id",
    "claims": "claim_id",
    "payments": "payment_id",
}

for name, df in data.items():
    pk = primary_keys[name]
    quality_rows.append({
        "table": name,
        "rows": len(df),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "missing_primary_keys": int(df[pk].isna().sum()) if pk in df else None,
        "duplicate_primary_keys": int(df[pk].duplicated().sum()) if pk in df else None,
    })

quality_report = pd.DataFrame(quality_rows)
display(quality_report)

,table,rows,duplicate_rows,missing_cells,missing_primary_keys,duplicate_primary_keys
0,customers,20000,0,0,0,0
1,vehicles,25000,0,0,0,0
2,policies,25000,0,59,0,0
3,claims,10000,0,25384,0,0
4,payments,7000,0,0,0,0


## 7. Data Cleaning

The reusable cleaning pipeline is implemented in `src/data_cleaner.py` and
orchestrated by `run_pipeline.py`. Raw files are preserved; cleaned files are
written to `data/cleaned/`. Missing workflow dates are not automatically invented.
Review the data-quality report and pipeline logs when anomalies are present.

In [6]:
for name, df in data.items():
    print(f"{name}: {len(df):,} cleaned rows")
    print("Missing cells:", int(df.isna().sum().sum()))

customers: 20,000 cleaned rows
Missing cells: 0
vehicles: 25,000 cleaned rows
Missing cells: 0
policies: 25,000 cleaned rows
Missing cells: 59
claims: 10,000 cleaned rows
Missing cells: 25384
payments: 7,000 cleaned rows
Missing cells: 0


## 8. Derived Columns

In [7]:
derived_columns = {
    "customers": ["customer_age"],
    "vehicles": ["vehicle_age"],
    "policies": ["policy_duration_days", "policy_active_flag"],
    "claims": [
        "claim_processing_days", "claim_settlement_days",
        "claim_to_premium_ratio", "claim_to_coverage_ratio",
        "paid_claim_ratio",
    ],
}

for table_name, columns in derived_columns.items():
    df = data.get(table_name, pd.DataFrame())
    print(f"\n{table_name}:")
    for column in columns:
        print(f"  {column}: {'present' if column in df.columns else 'not present'}")


customers:
  customer_age: present

vehicles:
  vehicle_age: present

policies:
  policy_duration_days: present
  policy_active_flag: present

claims:
  claim_processing_days: present
  claim_settlement_days: present
  claim_to_premium_ratio: present
  claim_to_coverage_ratio: present
  paid_claim_ratio: present


## 9. EDA — Univariate

In [8]:
policies = data["policies"]
claims = data["claims"]
vehicles = data["vehicles"]
customers = data["customers"]

for table_name, column, title in [
    ("policies", "premium_amount", "Premium Distribution"),
    ("claims", "claim_amount", "Claim Amount Distribution"),
    ("vehicles", "vehicle_age", "Vehicle Age Distribution"),
    ("customers", "customer_age", "Customer Age Distribution"),
]:
    df = data[table_name]
    if column in df:
        values = pd.to_numeric(df[column], errors="coerce").dropna()
        if not values.empty:
            ax = values.plot(kind="hist", bins=30, figsize=(8, 4), title=title)
            ax.set_xlabel(column.replace("_", " ").title())
            plt.show()
        else:
            print(f"No valid values for {table_name}.{column}")
    else:
        print(f"{table_name}.{column} not available")

for table_name, column in [
    ("policies", "policy_status"),
    ("claims", "claim_status"),
    ("claims", "claim_type"),
    ("vehicles", "vehicle_type"),
]:
    df = data[table_name]
    if column in df:
        display(df[column].fillna("Unknown").value_counts().rename("count").to_frame())

C:\Users\POTHURAJU\AppData\Local\Temp\ipykernel_6448\3326538775.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\POTHURAJU\AppData\Local\Temp\ipykernel_6448\3326538775.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\POTHURAJU\AppData\Local\Temp\ipykernel_6448\3326538775.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\POTHURAJU\AppData\Local\Temp\ipykernel_6448\3326538775.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,count
policy_status,
Expired,10464
Active,7551
Renewed,6665
Cancelled,320


,count
claim_status,
Approved,3790
Settled,3259
Pending,1791
Rejected,1160


,count
claim_type,
Accident,4838
Third Party Damage,2022
Theft,1585
Natural Disaster,796
Fire,759


,count
vehicle_type,
Car,25000


## 10. EDA — Bivariate

In [9]:
# Premium versus policy type
if {"policy_type", "premium_amount"}.issubset(policies.columns):
    display(policies.groupby("policy_type")["premium_amount"].agg(
        ["count", "mean", "median", "sum"]
    ).sort_values("sum", ascending=False))

# Claim amount versus vehicle type
claim_vehicle = viz._claim_customer_vehicle(data)
if {"vehicle_type", "claim_amount"}.issubset(claim_vehicle.columns):
    display(claim_vehicle.groupby("vehicle_type")["claim_amount"].agg(
        ["count", "mean", "median", "sum"]
    ).sort_values("mean", ascending=False))

# Region versus claim amount
region_col = "accident_location" if "accident_location" in claims.columns else None
if region_col and "claim_amount" in claims:
    display(claims.groupby(region_col)["claim_amount"].agg(
        ["count", "mean", "sum"]
    ).sort_values("count", ascending=False).head(15))

# Claim status versus settlement time
if {"claim_status", "claim_settlement_days"}.issubset(claims.columns):
    display(claims.groupby("claim_status")["claim_settlement_days"].agg(
        ["count", "mean", "median"]
    ))

,count,mean,median,sum
policy_type,,,,
Comprehensive,15085,"36,126.19","32,493.73","544,963,562.39"
Zero Depreciation,3700,"47,318.45","42,928.52","175,078,267.04"
Third Party,6215,"12,152.36","10,939.96","75,526,942.28"


,count,mean,median,sum
vehicle_type,,,,
Car,10000,"288,543.64","205,040.10","2,885,436,440.65"


,count,mean,sum
accident_location,,,
National Highway,1703,"289,409.17","492,863,811.03"
Parking Area,1687,"294,115.15","496,172,261.18"
City Road,1673,"287,990.25","481,807,680.50"
Highway,1673,"281,763.54","471,390,397.47"
Residential Area,1662,"292,001.64","485,306,727.79"
"Ambattur, Punjab",3,"121,431.97","364,295.90"
"Kirari Suleman Nagar, Mizoram",3,"125,076.86","375,230.59"
"Bongaigaon, Bihar",3,"191,658.84","574,976.53"
"Medininagar, Rajasthan",3,"391,246.87","1,173,740.61"


,count,mean,median
claim_status,,,
Approved,0,NaN,NaN
Pending,0,NaN,NaN
Rejected,0,NaN,NaN
Settled,3259,23.93,24.00


## 11. EDA — Multivariate

In [10]:
# Policy type, vehicle type, and claim amount
if {"policy_type", "vehicle_type", "claim_amount"}.issubset(claim_vehicle.columns):
    display(claim_vehicle.groupby(
        ["policy_type", "vehicle_type"]
    )["claim_amount"].agg(["count", "mean", "sum"]).reset_index())

# Region, claim type, and severity
if {"accident_location", "claim_type", "damage_severity"}.issubset(claims.columns):
    display(claims.groupby(
        ["accident_location", "claim_type", "damage_severity"]
    ).size().rename("claim_count").reset_index().sort_values(
        "claim_count", ascending=False
    ).head(20))

# Customer age, policy type, premium, and claims
if "customer_id" in customers and "customer_age" in customers:
    customer_age = customers[["customer_id", "customer_age"]].drop_duplicates("customer_id")
    if {"customer_id", "policy_type", "premium_amount"}.issubset(policies.columns):
        customer_policy = policies.merge(
            customer_age, on="customer_id", how="left", validate="many_to_one"
        )
        display(customer_policy.groupby("policy_type").agg(
            policies=("policy_id", "nunique"),
            average_premium=("premium_amount", "mean"),
            average_customer_age=("customer_age", "mean"),
        ))

,policy_type,vehicle_type,count,mean,sum
0,Comprehensive,Car,6011,"261,541.62","1,572,126,654.39"
1,Third Party,Car,2487,"355,490.51","884,104,889.55"
2,Zero Depreciation,Car,1502,"285,755.59","429,204,896.71"


,accident_location,claim_type,damage_severity,claim_count
558,Highway,Accident,Low,364
355,City Road,Accident,Medium,349
1310,Residential Area,Accident,Low,339
1090,National Highway,Accident,Medium,335
1181,Parking Area,Accident,Low,335
559,Highway,Accident,Medium,332
1089,National Highway,Accident,Low,324
354,City Road,Accident,Low,319
1182,Parking Area,Accident,Medium,317
1311,Residential Area,Accident,Medium,293


,policies,average_premium,average_customer_age
policy_type,,,
Comprehensive,15085,"36,126.19",43.86
Third Party,6215,"12,152.36",43.79
Zero Depreciation,3700,"47,318.45",43.66


## 12. Statistical Analysis

In [11]:
for table_name, column in [
    ("policies", "premium_amount"),
    ("claims", "claim_amount"),
    ("claims", "claim_settlement_days"),
]:
    df = data[table_name]
    if column in df:
        values = pd.to_numeric(df[column], errors="coerce").replace(
            [np.inf, -np.inf], np.nan
        ).dropna()
        if not values.empty:
            modes = values.mode()
            print(f"\n{table_name}.{column}")
            display(pd.Series({
                "count": values.count(),
                "mean": values.mean(),
                "median": values.median(),
                "mode": modes.iloc[0] if not modes.empty else np.nan,
                "variance": values.var(),
                "standard_deviation": values.std(),
                "minimum": values.min(),
                "maximum": values.max(),
            }).to_frame("value"))
        else:
            print(f"No valid observations: {table_name}.{column}")

display(pd.read_csv(ROOT / "reports" / "policy_numeric_correlations.csv"))
display(pd.read_csv(ROOT / "reports" / "claim_numeric_correlations.csv"))


policies.premium_amount


,value
count,"25,000.00"
mean,"31,822.75"
median,"28,917.67"
mode,"100,000.00"
variance,"327,588,325.69"
standard_deviation,"18,099.40"
minimum,"3,827.98"
maximum,"100,000.00"



claims.claim_amount


,value
count,"10,000.00"
mean,"288,543.64"
median,"205,040.10"
mode,"12,281.47"
variance,"69,194,304,007.55"
standard_deviation,"263,048.10"
minimum,"12,281.47"
maximum,"2,645,659.52"



claims.claim_settlement_days


,value
count,"3,259.00"
mean,23.93
median,24.00
mode,25.00
variance,96.96
standard_deviation,9.85
minimum,0.00
maximum,45.00


,Unnamed: 0,premium_amount,coverage_amount,policy_duration_days,policy_active_flag
0,premium_amount,1.00,0.40,0.00,0.01
1,coverage_amount,0.40,1.00,-0.00,0.00
2,policy_duration_days,0.00,-0.00,1.00,0.06
3,policy_active_flag,0.01,0.00,0.06,1.00


,Unnamed: 0,claim_amount,claim_processing_days,claim_settlement_days,premium_amount,coverage_amount,claim_to_premium_ratio,claim_to_coverage_ratio,total_payment_amount,paid_claim_ratio
0,claim_amount,1.00,-0.01,0.02,0.17,0.45,0.64,0.83,0.99,-0.02
1,claim_processing_days,-0.01,1.00,0.49,0.01,0.01,-0.02,-0.01,-0.01,0.00
2,claim_settlement_days,0.02,0.49,1.00,-0.00,0.01,0.00,0.01,0.02,0.00
3,premium_amount,0.17,0.01,-0.00,1.00,0.41,-0.41,-0.01,0.17,0.01
4,coverage_amount,0.45,0.01,0.01,0.41,1.00,0.21,-0.00,0.46,0.01
5,claim_to_premium_ratio,0.64,-0.02,0.00,-0.41,0.21,1.00,0.60,0.65,-0.01
6,claim_to_coverage_ratio,0.83,-0.01,0.01,-0.01,-0.00,0.60,1.00,0.82,-0.02
7,total_payment_amount,0.99,-0.01,0.02,0.17,0.46,0.65,0.82,1.00,0.07
8,paid_claim_ratio,-0.02,0.00,0.00,0.01,0.01,-0.01,-0.02,0.07,1.00


## 13. KPI Analysis

In [12]:
kpi_path = ROOT / "reports" / "kpi_summary.csv"
if kpi_path.exists():
    kpis = pd.read_csv(kpi_path)
    display(kpis)
else:
    print("KPI report not found. Run: python -m src.insurance_analysis")

,total_customers,total_policies,active_policies,expired_policies,cancelled_policies,renewed_policies,total_premium,average_premium,average_coverage,total_claims,approved_claims,rejected_claims,pending_claims,settled_claims,total_claim_amount,average_claim_amount,claim_approval_rate_pct,claim_rejection_rate_pct,average_claim_processing_days,average_claim_settlement_days,total_payment_amount,average_payment_amount,claim_to_premium_ratio,claim_to_coverage_ratio,paid_claim_ratio
0,20000,25000,7551,10464,320,6665,"795,568,771.71","31,822.75","1,127,529.40",10000,3790,1160,1791,3259,"2,885,436,440.65","288,543.64",37.90,11.60,8.31,23.93,"1,766,889,637.12","252,412.81",3.63,0.10,0.61


## 14. Business Questions

In [13]:
# These tables provide actual evidence for the required business questions.
if {"policy_type", "premium_amount"}.issubset(policies.columns):
    print("Policy types with the highest total premium:")
    display(policies.groupby("policy_type")["premium_amount"].sum().sort_values(
        ascending=False
    ).to_frame("total_premium"))

if {"claim_type", "claim_amount"}.issubset(claims.columns):
    print("Claim types with the highest average claim amount:")
    display(claims.groupby("claim_type")["claim_amount"].agg(
        ["count", "mean", "sum"]
    ).sort_values("mean", ascending=False))

if {"damage_severity", "claim_amount"}.issubset(claims.columns):
    print("Claim amount by damage severity:")
    display(claims.groupby("damage_severity")["claim_amount"].agg(
        ["count", "mean", "sum"]
    ).sort_values("mean", ascending=False))

if {"vehicle_type", "claim_amount"}.issubset(claim_vehicle.columns):
    print("Vehicle type claim patterns:")
    display(claim_vehicle.groupby("vehicle_type")["claim_amount"].agg(
        ["count", "mean", "sum"]
    ).sort_values("count", ascending=False))

if {"claim_status", "claim_settlement_days"}.issubset(claims.columns):
    print("Settlement time by claim status:")
    display(claims.groupby("claim_status")["claim_settlement_days"].agg(
        ["count", "mean", "median"]
    ))

Policy types with the highest total premium:


,total_premium
policy_type,
Comprehensive,"544,963,562.39"
Zero Depreciation,"175,078,267.04"
Third Party,"75,526,942.28"


Claim types with the highest average claim amount:


,count,mean,sum
claim_type,,,
Fire,759,"299,284.75","227,157,123.65"
Third Party Damage,2022,"294,464.14","595,406,487.74"
Natural Disaster,796,"289,044.59","230,079,497.02"
Theft,1585,"287,174.72","455,171,933.52"
Accident,4838,"284,750.19","1,377,621,398.72"


Claim amount by damage severity:


,count,mean,sum
damage_severity,,,
High,1814,"670,443.00","1,216,183,609.26"
Medium,4043,"309,307.82","1,250,531,524.54"
Low,4143,"101,067.18","418,721,306.85"


Vehicle type claim patterns:


,count,mean,sum
vehicle_type,,,
Car,10000,"288,543.64","2,885,436,440.65"


Settlement time by claim status:


,count,mean,median
claim_status,,,
Approved,0,NaN,NaN
Pending,0,NaN,NaN
Rejected,0,NaN,NaN
Settled,3259,23.93,24.00


## 15. Visualizations

In [14]:
chart_functions = [
    viz.plot_policy_status,
    viz.plot_policy_type_distribution,
    viz.plot_premium_by_policy_type,
    viz.plot_monthly_claims,
    viz.plot_claim_status,
    viz.plot_claim_type_distribution,
    viz.plot_claim_amount_distribution,
    viz.plot_claim_amount_by_vehicle_type,
    viz.plot_claims_by_region,
    viz.plot_claim_severity_by_region,
    viz.plot_average_settlement_time,
    viz.plot_claim_to_premium_by_policy_type,
    viz.plot_vehicle_age_vs_claim_amount,
    viz.plot_customer_age_vs_claim_amount,
    viz.plot_vehicle_type_claim_performance,
    viz.plot_damage_severity_distribution,
    viz.plot_payment_status,
    viz.plot_monthly_premium_trend,
]

for chart_function in chart_functions:
    fig = chart_function(data)
    display(fig)
    plt.close(fig)

<Figure size 1000x500 with 1 Axes>

<Figure size 1000x500 with 1 Axes>

<Figure size 1000x500 with 1 Axes>

<Figure size 1100x500 with 1 Axes>

<Figure size 1000x500 with 1 Axes>

...

<Figure size 1000x500 with 1 Axes>

<Figure size 1000x500 with 1 Axes>

<Figure size 1000x500 with 1 Axes>

<Figure size 1000x500 with 1 Axes>

<Figure size 1100x500 with 1 Axes>

## 16. Evidence-Based Insights

In [15]:
insight_functions = [
    insight_engine.generate_portfolio_insights,
    insight_engine.generate_claim_insights,
    insight_engine.generate_premium_insights,
    insight_engine.generate_customer_insights,
    insight_engine.generate_vehicle_insights,
    insight_engine.generate_risk_patterns,
    insight_engine.generate_business_recommendations,
]

for function in insight_functions:
    print(f"\n### {function.__name__}")
    results = function(data)
    if not results:
        print("No insight generated: required data or valid observations were unavailable.")
    else:
        display(pd.DataFrame(results))


### generate_portfolio_insights


,pattern,evidence,business_meaning,recommendation
0,Expired is the most frequent policy status.,"10,464 of 25,000 policies (41.86%).",The status mix describes the current compositi...,Review status definitions and compare the port...
1,Comprehensive has the highest policy count.,"15,085 policies.",Policy volume is concentrated in this product ...,Review service capacity and performance by pol...



### generate_claim_insights


,pattern,evidence,business_meaning,recommendation
0,Approved is the most common recorded claim sta...,"3,790 of 10,000 claims (37.90%).",The distribution indicates where claims are co...,Review workflow definitions and investigate an...
1,Fire has the highest average claim amount amon...,"Average claim amount: 299,284.75; records: 759.",This category may deserve closer review of cla...,"Inspect sample claims and compare severity, ex..."
2,Low is the most frequently recorded damage sev...,"4,143 claims.",This shows the most common recorded severity c...,Compare severity categories with claim amounts...



### generate_premium_insights


...

,pattern,evidence,business_meaning,recommendation
0,Car has the highest average claim amount among...,"Average claim amount: 288,543.64; claims: 10,000.",Average claim severity can differ from total c...,"Compare claim frequency, sample sizes, vehicle..."
1,Petrol vehicles have the largest recorded clai...,"4,578 claims.",Raw counts can reflect how common a fuel type ...,Compare claim counts with the number of insure...



### generate_risk_patterns


,pattern,evidence,business_meaning,recommendation
0,Premium values vary across the policy portfolio.,"Mean premium: 31,822.75; median premium: 28,91...",A gap between mean and median can indicate a s...,Review the premium distribution and compare pr...
1,Third Party has the highest observed claim-amo...,Ratio of summed claim amounts to summed premiu...,This descriptive comparison may indicate a cat...,"Validate exposure periods, policy mix, claim m..."
2,Car has the highest average claim amount among...,"Average claim amount: 288,543.64; claims: 10,000.",Average claim severity can differ from total c...,"Compare claim frequency, sample sizes, vehicle..."
3,Petrol vehicles have the largest recorded clai...,"4,578 claims.",Raw counts can reflect how common a fuel type ...,Compare claim counts with the number of insure...
4,Approved is the most common recorded claim sta...,"3,790 of 10,000 claims (37.90%).",The distribution indicates where claims are co...,Review workflow definitions and investigate an...
5,Fire has the highest average claim amount amon...,"Average claim amount: 299,284.75; records: 759.",This category may deserve closer review of cla...,"Inspect sample claims and compare severity, ex..."
6,Low is the most frequently recorded damage sev...,"4,143 claims.",This shows the most common recorded severity c...,Compare severity categories with claim amounts...



### generate_business_recommendations


,pattern,evidence,business_meaning,recommendation
0,Expired is the most frequent policy status.,"10,464 of 25,000 policies (41.86%).",The status mix describes the current compositi...,Review status definitions and compare the port...
1,Comprehensive has the highest policy count.,"15,085 policies.",Policy volume is concentrated in this product ...,Review service capacity and performance by pol...
2,Customers in age group 56–65 have the highest ...,Average claims per customer: 0.510.,This is a descriptive segmentation result and ...,Compare group sizes and policy exposure before...
3,Premium values vary across the policy portfolio.,"Mean premium: 31,822.75; median premium: 28,91...",A gap between mean and median can indicate a s...,Review the premium distribution and compare pr...
4,Third Party has the highest observed claim-amo...,Ratio of summed claim amounts to summed premiu...,This descriptive comparison may indicate a cat...,"Validate exposure periods, policy mix, claim m..."
5,Car has the highest average claim amount among...,"Average claim amount: 288,543.64; claims: 10,000.",Average claim severity can differ from total c...,"Compare claim frequency, sample sizes, vehicle..."
6,Petrol vehicles have the largest recorded clai...,"4,578 claims.",Raw counts can reflect how common a fuel type ...,Compare claim counts with the number of insure...
7,Approved is the most common recorded claim sta...,"3,790 of 10,000 claims (37.90%).",The distribution indicates where claims are co...,Review workflow definitions and investigate an...
8,Fire has the highest average claim amount amon...,"Average claim amount: 299,284.75; records: 759.",This category may deserve closer review of cla...,"Inspect sample claims and compare severity, ex..."
9,Low is the most frequently recorded damage sev...,"4,143 claims.",This shows the most common recorded severity c...,Compare severity categories with claim amounts...


## Interpretation and Limitations

- Findings describe the supplied datasets and do not establish causation.
- Claim amounts are not the same as actual payment amounts.
- A claim-to-premium ratio is a descriptive comparison unless earned premium,
  exposure periods, and an approved actuarial definition are available.
- Missing workflow dates should not be invented.
- Review anomalous policy dates and data-quality findings before making decisions.